# ALD-KG Pipeline
각 셀을 순서대로 실행하세요. 스텝별로 독립 실행 가능합니다.

In [ ]:
# 공통 설정 확인
from config import PDF_DIR, OUTPUT_DIR, STEP
print('PDF_DIR   :', PDF_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('PDFs found:', sorted(PDF_DIR.glob('*.pdf')))

PDF_DIR   : /home/ftk3187/github/PSED/0604_kg/pdf
OUTPUT_DIR: /home/ftk3187/github/PSED/0604_kg/output
PDFs found: [PosixPath('/home/ftk3187/github/PSED/0604_kg/pdf/Arts et al. - 2019 - Film Conformality and Extracted Recombination Probabilities of O Atoms during Plasma-Assisted Atomic.pdf')]


## Step 01 — PDF → text / figures / tables (Docling)

In [7]:
from importlib import import_module
m01 = import_module('01_docling_extract')
m01.main()

/home/ftk3187/miniconda3/envs/psed310/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:docling-extract:Converting: /home/ftk3187/github/PSED/0604_kg/pdf/Arts et al. - 2019 - Film Conformality and Extracted Recombination Probabilities of O Atoms during Plasma-Assisted Atomic.pdf
INFO:docling.datamodel.document:detected formats: [<InputFormat.PDF: 'pdf'>]
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.document_converter:Initializing pipeline for StandardPdfPipeline with options hash badc7e69fd629b6fbcb52cb6de87ea0c
INFO:docling.models.factories.base_factory:Loading plugin 'docling_defaults'
INFO:docling.models.factories:Registered picture descriptions: ['picture_description_vlm_engine', 'vlm', 'api']
INFO:docling.models.factories.base_factory:Loading plugin 'docling_de

(scale: float = 1.0, max_size: Optional[int] = None, cropbox: Optional[docling_core.types.doc.base.BoundingBox] = None) -> Optional[PIL.Image.Image]

===== DEBUG CAPTION STRUCTURE =====
Total pictures: 10

Picture 1
caption field: []

Picture 2
caption field: []

Picture 3
caption field: []

Picture 4
caption field: []

Picture 5
caption field: []

Picture 6
caption field: []

Picture 7
caption field: [RefItem(cref='#/texts/41')]

Picture 8
caption field: [RefItem(cref='#/texts/98')]

Picture 9
caption field: []

Picture 10
caption field: [RefItem(cref='#/texts/140')]

Doc captions count: 0



INFO:docling-extract:Done: Arts et al. - 2019 - Film Conformality and Extracted Recombination Probabilities of O Atoms during Plasma-Assisted Atomic (tables=1, figures=10, formulas=6)


## Step 02 — Figure filtering (Vision LLM)

In [1]:
from config import OUTPUT_DIR
from importlib import import_module
m02 = import_module('02_figure_filter')
for paper_dir in sorted(OUTPUT_DIR.iterdir()):
    if paper_dir.is_dir():
        print(f'\nProcessing {paper_dir.name}')
        m02.filter_figures(paper_dir.name, OUTPUT_DIR)


Processing Arts et al. - 2019 - Film Conformality and Extracted Recombination Probabilities of O Atoms during Plasma-Assisted Atomic
VISION RAW RESPONSE:
{
  "is_scientific_figure": true,
  "confidence": 1.0,
  "reason": "The image is a data plot with labeled axes ('Normalized thickness', 'Distance (µm)', 'Distance / cavity height'), multiple data curves representing different materials (SiO2, TiO2, Al2O3, HfO2) and plasma exposure times, and a legend. It is labeled as panel 'B)' and the caption explicitly refers to it as showing 'measured thickness profiles', which are scientific data visualizations."
}
KEEP: figure-009.png
VISION RAW RESPONSE:
{
  "is_scientific_figure": true,
  "confidence": 1.0,
  "reason": "The image is a data plot with labeled axes, numerical scales, data points with error bars, linear fit lines, a legend, and an equation, all characteristic of a scientific figure presenting experimental results and analysis. The caption also explicitly refers to it as 'Figure 3

## Step 03 — Plot → data (Gemini Vision)

In [1]:
from config import OUTPUT_DIR
from importlib import import_module
m03 = import_module('03_plot_to_data')
for paper_dir in sorted(OUTPUT_DIR.iterdir()):
    if paper_dir.is_dir():
        m03.process_folder(paper_dir.name, OUTPUT_DIR)

API Key loaded: AIza...VbfI

Processing: Arts et al. - 2019 - Film Conformality and Extracted Recombination Probabilities of O Atoms during Plasma-Assisted Atomic (2 images)
Processing: figure-009.png
  → 8 series parsed
  Saved: /home/ftk3187/github/PSED/0604_kg/output/Arts et al. - 2019 - Film Conformality and Extracted Recombination Probabilities of O Atoms during Plasma-Assisted Atomic/03_plot_to_data/figure-009.json
Processing: figure-010.png
  → 4 series parsed
  Saved: /home/ftk3187/github/PSED/0604_kg/output/Arts et al. - 2019 - Film Conformality and Extracted Recombination Probabilities of O Atoms during Plasma-Assisted Atomic/03_plot_to_data/figure-010.json


## Step 04 — Equations → data (Gemini Vision)

In [10]:
from config import OUTPUT_DIR
from importlib import import_module

m04=import_module("04_formula_to_data")

for paper_dir in sorted(OUTPUT_DIR.iterdir()):
    if paper_dir.is_dir():
        m04.process_equations(paper_dir.name,OUTPUT_DIR)

INFO:google_genai.models:AFC is enabled with max remote calls: 10.


KeyboardInterrupt: 

## Step 05 — Paper Schema

In [6]:
from importlib import import_module
m05 = import_module('05_paper_schema')
m05.main()

INFO:google_genai.models:AFC is enabled with max remote calls: 10.


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


[OK] Arts et al. - 2019 - Sticking probabilities of H2O and Al(CH3)3 during atomic layer deposition of Al2O3 extracted from th


## Step 06 — Enrich Figures

In [1]:
from importlib import import_module
m06 = import_module('06_enrich_figures')
m06.main()

[PROCESS] figure-009 figure-009.json
[PROCESS] figure-010 figure-010.json


## Step 07 — Experiment Schema (Gemini)

In [1]:
from importlib import import_module
m07 = import_module('07_experiment_schema')
m07.main()

  [FIGURE] figure-009.json
    [LLM] calling... series=['HfO₂ (120 s)', 'Al₂O₃ (120 s)', 'SiO₂ (3.8 s)', 'TiO₂ (12 s)', 'SiO₂ (12 s)', 'SiO₂ (38 s)', 'TiO₂ (120 s)', 'SiO₂ (120 s)']
    [LLM] got 8 experiments
    [OK] experiment-00001
    [OK] experiment-00002
    [OK] experiment-00003
    [OK] experiment-00004
    [OK] experiment-00005
    [OK] experiment-00006
    [OK] experiment-00007
    [OK] experiment-00008
  [FIGURE] figure-010.json
    [LLM] calling... series=['SiO₂', 'TiO₂', 'Linear Fit SiO₂', 'Linear Fit TiO₂']
    [LLM] got 4 experiments
    [OK] experiment-00009
    [OK] experiment-00010
    [OK] experiment-00011
    [OK] experiment-00012
[DONE] Arts et al. - 2019 - Film Conformality and Extracted Recombination Probabilities of O Atoms during Plasma-Assisted Atomic: 12 experiments


## Step 08 — Normalize

In [1]:
from importlib import import_module
m08 = import_module('08_normalize')
m08.main()

[OK] experiment-00001.json
[OK] experiment-00002.json
[OK] experiment-00003.json
[OK] experiment-00004.json
[OK] experiment-00005.json
[OK] experiment-00006.json
[OK] experiment-00007.json
[OK] experiment-00008.json
[OK] experiment-00009.json
[OK] experiment-00010.json
[OK] experiment-00011.json
[OK] experiment-00012.json
[DONE] Arts et al. - 2019 - Film Conformality and Extracted Recombination Probabilities of O Atoms during Plasma-Assisted Atomic: 12 experiments normalized
[OK] experiment-00001.json
[OK] experiment-00002.json
[OK] experiment-00003.json
[OK] experiment-00004.json
[OK] experiment-00005.json
[OK] experiment-00006.json
[OK] experiment-00007.json
[OK] experiment-00008.json
[OK] experiment-00009.json
[OK] experiment-00010.json
[OK] experiment-00011.json
[OK] experiment-00012.json
[OK] experiment-00013.json
[OK] experiment-00014.json
[OK] experiment-00015.json
[OK] experiment-00016.json
[OK] experiment-00017.json
[OK] experiment-00018.json
[DONE] Yim et al. - 2020 - Saturat

In [ ]:
from importlib import import_module
m10 = import_module('10_match')
m10.main()

[INFO] loaded 34 experiments from /home/ftk3187/github/PSED/0604_kg/output
[INFO] total pairs:       74
[INFO] cross-paper pairs: 18
[INFO] top score:         1.000  (experiment-00002 ↔ experiment-00011)
[DONE] saved → /home/ftk3187/github/PSED/0604_kg/output/matches.json


In [3]:
from importlib import import_module
m11 = import_module('11_kg')
m11.main()

[INFO] 22 experiments, 74 pairs loaded

[GRAPH STATS]
  nodes: 36
    experiment          : 22
    material            : 2
    paper               : 2
    variable            : 10
  edges: 220
    from_paper          : 22
    has_ctrl            : 52
    has_dep             : 25
    has_indep           : 28
    similar_to          : 71
    uses_material       : 22

  similar_to edges (>= 0.5):
    total:       71
    cross-paper: 17
    score max:   1.000
    score mean:  0.742

  top cross-paper pairs:
    experiment-00002 ↔ experiment-00024  score=1.0  dep=film_thickness
    experiment-00002 ↔ experiment-00025  score=1.0  dep=film_thickness
    experiment-00011 ↔ experiment-00024  score=1.0  dep=film_thickness
    experiment-00011 ↔ experiment-00025  score=1.0  dep=film_thickness
    experiment-00002 ↔ experiment-00016  score=0.8333  dep=normalized_thickness

[DONE] GraphML → /home/ftk3187/github/PSED/0604_kg/output/knowledge_graph.graphml
[DONE] JSON    → /home/ftk3187/github/PSED/0

In [4]:
from importlib import import_module
m11 = import_module('visualize_kg')
m11.main()

[INFO] nodes=36, edges=220
[DONE] → /home/ftk3187/github/PSED/0604_kg/output/kg.html
       브라우저에서 열어주세요: open /home/ftk3187/github/PSED/0604_kg/output/kg.html


In [5]:
from importlib import import_module
m11 = import_module('visualize_matches')
m11.main()

[INFO] 22 experiments, 74 matches
[DONE] matrix → /home/ftk3187/github/PSED/0604_kg/output/viz_matrix.html
[DONE] viewer → /home/ftk3187/github/PSED/0604_kg/output/viz_viewer.html

브라우저에서 열기:
  open /home/ftk3187/github/PSED/0604_kg/output/viz_matrix.html
  open /home/ftk3187/github/PSED/0604_kg/output/viz_viewer.html


## Output 구조 확인

In [ ]:
from config import OUTPUT_DIR, STEP
for paper_dir in sorted(OUTPUT_DIR.iterdir()):
    if not paper_dir.is_dir(): continue
    print(f'\n[{paper_dir.name}]')
    for step_name in STEP.values():
        step_path = paper_dir / step_name
        if step_path.exists():
            files = list(step_path.rglob('*'))
            print(f'  {step_name}: {len(files)} files')
        else:
            print(f'  {step_name}: (not run)')


[Aguinsky et al. - 2023 - Modeling incomplete conformality during atomic layer deposition in high aspect ratio structures]
  01_docling: 20 files
  02_figure_filter: 13 files
  03_plot_to_data: 7 files
  04_build_segment: 1 files
  05_sentence_tagging: 3 files
  06_evidence_pools: 3 files
  07_schema_extraction: 3 files

[Arts et al. - 2019 - Sticking probabilities of H2O and Al(CH3)3 during atomic layer deposition of Al2O3 extracted from th]
  01_docling: 18 files
  02_figure_filter: 13 files
  03_plot_to_data: 3 files
  04_build_segment: 1 files
  05_sentence_tagging: 3 files
  06_evidence_pools: 3 files
  07_schema_extraction: 3 files

[Yanguas-Gil and Elam - 2012 - Simple model for atomic layer deposition precursor reaction and transport in a viscous-flow tubular]
  01_docling: 20 files
  02_figure_filter: 17 files
  03_plot_to_data: 8 files
  04_build_segment: 1 files
  05_sentence_tagging: 3 files
  06_evidence_pools: 3 files
  07_schema_extraction: 3 files

[Yim et al. - 2020 -

: 